# LastDance · 3-ID end-to-end smoke test
Enable a **GPU** and **Internet** on Kaggle. Attach the released LifeMate training data with audio and simulator files, the final Search-First A100 checkpoint-100 adapter, and the fine-tuned Gipformer ONNX export. Set paths in the configuration cell if auto-detection is ambiguous. Qwen3.5-4B downloads from Hugging Face when `MODEL_PATH` is empty.

For a private clone, add a Kaggle Secret named `GITHUB_TOKEN` with read access and set `PRIVATE_REPO = True`. Each case runs through the submission entrypoint and is scored locally against gold replay after prediction.

In [ ]:
from pathlib import Path
import hashlib
import json
import os
import subprocess
import sys
import tempfile

REPO_URL = "https://github.com/AmnO-O/LastDance.git"
REF = "main"
PRIVATE_REPO = False
DATA_DIR = ""      # Optional: path to the train split itself
ADAPTER_PATH = ""  # Optional: directory with adapter_config.json and adapter_model.safetensors
ASR_PATH = ""      # Optional: directory with the four Gipformer files
MODEL_PATH = ""    # Optional: local Qwen3.5-4B directory

IDS = ["train_0459", "train_1409", "train_1542"]
INPUT = Path("/kaggle/input")
WORK = Path("/kaggle/working")
REPO = WORK / "LastDance"
OUTPUT = WORK / "lastdance_smoke"
OUTPUT.mkdir(parents=True, exist_ok=True)


In [ ]:
env = dict(os.environ, GIT_TERMINAL_PROMPT="0")
if PRIVATE_REPO:
    from kaggle_secrets import UserSecretsClient

    env["LASTDANCE_GITHUB_TOKEN"] = UserSecretsClient().get_secret("GITHUB_TOKEN")

with tempfile.TemporaryDirectory() as tmp:
    if PRIVATE_REPO:
        askpass = Path(tmp) / "git-askpass"
        askpass.write_text(
            '#!/bin/sh\ncase "$1" in\n*Username*) echo x-access-token ;;\n'
            '*) printf "%s\\n" "$LASTDANCE_GITHUB_TOKEN" ;;\nesac\n',
            encoding="utf-8",
        )
        askpass.chmod(0o700)
        env["GIT_ASKPASS"] = str(askpass)

    if not REPO.exists():
        subprocess.run(["git", "clone", REPO_URL, str(REPO)], env=env, check=True)
    subprocess.run(["git", "-C", str(REPO), "fetch", "origin", REF], env=env, check=True)
    subprocess.run(["git", "-C", str(REPO), "checkout", "--detach", "FETCH_HEAD"], check=True)

commit = subprocess.check_output(["git", "-C", str(REPO), "rev-parse", "HEAD"], text=True).strip()
env.pop("LASTDANCE_GITHUB_TOKEN", None)
print(f"LastDance commit {commit}")


In [ ]:
ADAPTER_SHA256 = "927186df0ff6c7bb9deca0f1a9f960d14ff6f246f7c101e16069d7e974c9f8c8"

def digest(path):
    with path.open("rb") as source:
        return hashlib.file_digest(source, "sha256").hexdigest()

def choose_path(override, marker, valid, label):
    if override:
        candidate = Path(override)
        if not valid(candidate):
            raise ValueError(f"Invalid {label} directory: {candidate}")
        return candidate
    matches = sorted({p.parent for p in INPUT.rglob(marker) if valid(p.parent)})
    if len(matches) != 1:
        raise ValueError(f"Expected one {label} directory, found {matches}. Set its path in the config cell.")
    return matches[0]

def valid_adapter(path):
    weights = path / "adapter_model.safetensors"
    return (path / "adapter_config.json").is_file() and weights.is_file() and digest(weights) == ADAPTER_SHA256

def valid_asr(path):
    return (path / "tokens.txt").is_file() and all(
        any((path / f"{name}{suffix}").is_file() for suffix in (".onnx", ".int8.onnx"))
        for name in ("encoder", "decoder", "joiner")
    )

def valid_data(path):
    return path.name == "train" and all(
        (path / name).exists() for name in (
            "train.jsonl", "train_tools_catalog.json", "train_environment", "zaic_tools/__init__.py"
        )
    )

train_dir = choose_path(DATA_DIR, "train_tools_catalog.json", valid_data, "train split")
adapter = choose_path(ADAPTER_PATH, "adapter_model.safetensors", valid_adapter, "final adapter")
asr = choose_path(ASR_PATH, "tokens.txt", valid_asr, "Gipformer")

sys.path.insert(0, str(REPO))
sys.path.insert(0, str(REPO / "src"))
with (train_dir / "train.jsonl").open(encoding="utf-8") as source:
    by_id = {row["id"]: row for row in map(json.loads, filter(str.strip, source)) if row["id"] in IDS}
samples = [by_id[sid] for sid in IDS]
for row in samples:
    if not row.get("audio"):
        raise ValueError(f"{row['id']} has no current-turn audio")
    turns = [row, *(turn for turn in row.get("history", []) if turn.get("role") == "user" and turn.get("audio"))]
    for turn in turns:
        path = train_dir / turn["audio"]
        if not path.is_file():
            raise FileNotFoundError(path)
print("Inputs verified for", IDS)


In [ ]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError("Enable a GPU accelerator before running inference.")
print(torch.cuda.get_device_name(0))
subprocess.run([
    sys.executable, "-m", "pip", "install", "--upgrade",
    "transformers>=5", "peft", "accelerate", "sherpa-onnx", "soundfile", "sentence-transformers",
], check=True)


In [ ]:
predictions_path = OUTPUT / "submission.jsonl"
command = [
    sys.executable, str(REPO / "submission/predict.py"),
    "--data-dir", str(train_dir), "--output", str(predictions_path),
    "--asr-path", str(asr), "--adapter-path", str(adapter), "--ids", *IDS,
]
command += ["--model-path", MODEL_PATH] if MODEL_PATH else ["--allow-download"]
inference_env = dict(os.environ, HF_HUB_OFFLINE="0", TRANSFORMERS_OFFLINE="0")
with (OUTPUT / "inference.log").open("w", encoding="utf-8") as log:
    with subprocess.Popen(
        command, cwd=REPO, env=inference_env, stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT, encoding="utf-8", errors="replace",
    ) as process:
        for line in process.stdout:
            print(line, end="")
            log.write(line)
        if process.wait():
            raise RuntimeError(f"Predictor failed. See {log.name}.")

rows = [json.loads(line) for line in predictions_path.read_text(encoding="utf-8").splitlines()]
if [row["id"] for row in rows] != IDS:
    raise RuntimeError("Predictions must contain exactly the three requested IDs in order.")


In [ ]:
from lifemate.catalog import Catalog
from lifemate.data import Sample
from lifemate.runtime import make_tool_env
from lifemate.scoring import evaluate_predictions

root = train_dir.parent
catalog = Catalog.load(train_dir / "train_tools_catalog.json")
predictions = {row["id"]: row for row in rows}
report = evaluate_predictions(
    [Sample(raw=row, split="train", root=root) for row in samples],
    predictions,
    catalog,
    lambda sample_id: make_tool_env(root, "train", sample_id),
)
report["commit"] = commit
report["adapter_sha256"] = ADAPTER_SHA256
for detail in report["details"]:
    detail["time_ms"] = predictions[detail["id"]]["time"]
(OUTPUT / "report.json").write_text(json.dumps(report, indent=2), encoding="utf-8")
from IPython.display import display
import pandas as pd
display(pd.DataFrame(report["details"])[["id", "score", "hash_ok", "predicted", "gold", "time_ms"]])
print(f"Score: {report['score_100']:.2f}/100")
print(f"Artifacts: {OUTPUT}")
